# 13.04 · The voice agent sandwich: STT → support agent → TTS (batch version)

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 13.02–13.03 and Project 09 (the Orbita Cloud support desk). Read the module README first. |
| 🔑 Keys | optional: `GROQ_API_KEY` (Whisper, the desk's model, Orpheus), `GOOGLE_API_KEY` (Gemini TTS, free tier) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Compose** speech-to-text, a tool-calling agent and text-to-speech into one voice turn, and time every stage.
2. **Explain** the latency a caller feels: from the end of their question to the first sound of the answer.
3. **Overlap** the agent and TTS by streaming the reply and speaking each sentence as soon as it is written.
4. **Shorten** an agent's reply for listening: no citations, no headings, three sentences at most.
5. **Identify** the caller from the phone system, never from what they say, and see the desk refuse a spoken ID.
6. **Compare** this batch sandwich with LangChain's official streaming voice agent and list what it leaves out.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Build a voice agent with LangChain](https://docs.langchain.com/oss/python/langchain/voice-agent) | The sandwich architecture, streamed end to end over WebSockets (AssemblyAI + Cartesia, under 700 ms claimed) |
| [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) | `stream_mode="messages"`: the agent's tokens as they are generated |
| [Voice tracing fundamentals](https://docs.langchain.com/langsmith/trace-voice-fundamentals) | How to trace a voice turn as one run with stage children |
| [Optimizing latency on Groq](https://console.groq.com/docs/production-readiness/optimizing-latency) | Time to first token, model size, prompt length |

> **Why this matters at work.** A phone line that answers "where is my invoice?" without a queue is the most requested voice feature in support. The parts are the ones you already built: Whisper (13.02), the Project 09 desk, TTS (13.03). The engineering is in **joining** them: latency, identity and what to do when one part fails.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "groq>=0.37.1,<1" "google-genai>=2.25.0,<3" "langgraph>=1.2.14,<2" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "groq", "google-genai",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm13-voice-agent-sandwich'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from IPython.display import Audio, display

AUDIO_DIR = REPO_ROOT / "data" / "outputs" / "m13" / "audio"   # git-ignored
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Three calls to the Orbita Cloud phone line (synthetic). The phone system knows the caller's number.
CALLS = {
    "q-invoices": {"text": "Hi, this is customer C-1002. What are my latest invoices?", "voice": "Kore", "caller_number": "+57 300 555 0142"},
    "q-status":   {"text": "Is the station sync service down right now? Our dashboard has no new readings.", "voice": "Puck", "caller_number": "+57 310 555 0199"},
    "q-unlock":   {"text": "Our team is locked out of the account since the password reset. Customer C-1002, please unlock it.", "voice": "Fenrir", "caller_number": "+57 315 555 0101"},
}
print(len(CALLS), "calls")

---
## Part 1 · The sandwich and its latency budget

```
caller speaks ──► [STT] ──► text ──► [agent + tools] ──► reply ──► [TTS] ──► caller hears
               end of speech                                              first audio
```

What the caller feels is the gap between the **end of their question** and the **first sound** of the answer. Conversation research and the voice-agent docs put a comfortable gap under about one second; above two to three seconds callers start saying "hello?". Every stage takes its share:

In [ ]:
# 🧪 Offline: a latency budget for one voice turn (targets, not measurements).
budget = [{"stage": "STT (whole question)", "target s": 0.5}, {"stage": "agent until first sentence", "target s": 1.0},
          {"stage": "TTS first sentence", "target s": 0.5}]
table(budget + [{"stage": "first audio (sum)", "target s": sum(b["target s"] for b in budget)}], ["stage", "target s"])

**You should now see** a 2.0-second budget: 0.5 for STT, 1.0 for the agent, 0.5 for the first TTS chunk. Parts 4–5 measure against it.

---
## Part 2 · Record the three calls (Gemini TTS, once)

As in Lesson 13.02, the callers are synthesized once and read from disk afterwards.

In [ ]:
# 💸 Gemini TTS: three short questions, the first time only.
from mastery.voice import cached_audio, gemini_tts, read_wav

clips = {}
for call_id, call in CALLS.items():
    path = AUDIO_DIR / f"{call_id}.wav"
    if path.is_file() or HAVE_GOOGLE_KEY:
        data, from_cache = cached_audio(path, lambda call=call: gemini_tts(call["text"], voice=call["voice"], sample_rate=16_000))
        clips[call_id] = data
        print(f"{call_id:<11} {read_wav(data).seconds:4.1f} s", "(from disk)" if from_cache else "(synthesized)")
if not clips:
    print("⏭️ Skipped: GOOGLE_API_KEY is not set and no recordings exist yet; the calls will be SIMULATED.")

> 💸 Three Gemini TTS requests (free tier), once. Without `GOOGLE_API_KEY` and no saved files it prints `⏭️ Skipped`.

**You should now see** three lines of 3 to 6 seconds each, `(synthesized)` the first time.

---
## Part 3 · The whole turn, offline first

`VoicePipeline` takes three parts with small interfaces: a transcriber (`transcribe`), an agent (`invoke` and `stream`) and a list of speakers tried in order. Offline: `FakeTranscriber`, the Project 09 desk with its rule-based `offline_model`, and `FakeSpeaker`. `GraphAgent` wraps the desk so each caller gets their own thread.

In [ ]:
# 🧪 Offline: one full turn with the offline desk (real tools, rule-based model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportContext, SupportDesk, build_single_agent_desk, make_support_tools, offline_model
from mastery.voice import FakeSpeaker, FakeTranscriber, GraphAgent, VoicePipeline

desk = SupportDesk.load()
TOOLS = make_support_tools(desk)
offline_desk = build_single_agent_desk(offline_model(), TOOLS, checkpointer=InMemorySaver())
fake_stt = FakeTranscriber(lambda key: {"text": CALLS[key]["text"].replace("C-1002", "C1002")})   # SIMULATED Whisper

from mastery.voice import fix_ids

for clean in (None, fix_ids):
    pipe = VoicePipeline(fake_stt, GraphAgent(offline_desk), [FakeSpeaker(max_chars=2000)], clean_transcript=clean)
    turn = pipe.run(None, key="q-invoices")
    print("clean  :", clean.__name__ if clean else "(none)")
    print("heard  :", turn.transcript.text)
    print("reply  :", turn.reply[:160])
    print("spoken :", turn.spoken)
    print("timings:", turn.timings, "(SIMULATED stages)\n")

**You should now see** two turns. Without cleaning, Whisper's `C1002` (no hyphen, a typical slip) makes the desk ask *Could you give me your customer ID?*. With `fix_ids` the desk replies `Invoices for C-1002 (Team plan): INV-2001 2026-10-03 $199.00 …`, spoken as `C-1-0-0-2` and `199 dollars`. Timings are near zero (nothing real ran).

### What just happened?

A one-line repair between STT and the agent saved a whole extra turn ("what's your customer ID?"), which on the phone costs 10 seconds and some patience. Fix **known** slips deterministically and leave the rest to the agent. Then the desk answered from its **tools** (it looked up the invoices) even though the input came from speech: the voice layer is a wrapper, not a new agent. `for_voice` turned the reply into something to listen to: three sentences at most, no file citations or headings, IDs and prices said aloud.

---
## Part 4 · Live: Whisper → the desk on `gpt-oss-20b` → Gemini TTS

The same pipeline with real parts. Speakers: Gemini first (free tier), Orpheus as backup, text if both fail.

In [ ]:
# 💸 (LIVE) Three calls through the real sandwich, sequential mode.
from mastery.voice import GeminiSpeaker, GroqTranscriber, OrpheusSpeaker, TtsBudget
from mastery.voice.tts import ORPHEUS_TERMS_URL

LIVE = HAVE_GROQ_KEY and len(clips) == len(CALLS)
ORPHEUS_BUDGET = TtsBudget()
speakers = ([GeminiSpeaker(style="calm, friendly support agent")] if HAVE_GOOGLE_KEY else []) + \
           ([OrpheusSpeaker(budget=ORPHEUS_BUDGET)] if HAVE_GROQ_KEY else [])
if LIVE:
    from mastery.models import get_model

    from mastery.voice import fix_ids

    live_desk = build_single_agent_desk(get_model("fast", temperature=0), TOOLS, checkpointer=InMemorySaver())
    stt = GroqTranscriber()
    HINTS = {"prompt": "Orbita Cloud support. Customer IDs like C-1002. Station sync.", "language": "en"}
    rows, turns = [], {}
    for call_id in CALLS:
        live = VoicePipeline(stt, GraphAgent(live_desk), speakers, stt_options=HINTS, clean_transcript=fix_ids)
        turns[call_id] = t = live.run(clips[call_id])
        rows.append({"call": call_id, **t.timings, "provider": t.speech.provider})
        print(f"{call_id}: {t.spoken}")
    table(rows, ["call", "stt", "agent", "tts", "first_audio", "total", "provider"])
    display(Audio(turns["q-invoices"].speech.wav))
else:
    print("⏭️ Skipped: needs GROQ_API_KEY and the Part 2 recordings (see the setup cell).")

> 💸 Per call: one Whisper request, two to four `gpt-oss-20b` calls (the desk routes, then uses a tool), one Gemini TTS request. About 6K tokens in total.

**You should now see** three spoken answers (C-1002's invoices; station sync degraded until 20:00 UTC; the unlock result) and a table where **agent** is the largest stage (one to three seconds: routing plus a tool call), STT a few tenths, and TTS one to two seconds. `first_audio` is typically 2.5–5 s: over the Part 1 budget.

### What just happened?

In sequential mode the caller waits for **all** of: transcription, every agent step, and the whole TTS clip. The agent is the biggest share, and you can't make tool calls disappear; but you can stop waiting for the *whole* reply before speaking. That is the next part.

---
## Part 5 · Speak each sentence as soon as it is written

With `mode="sentences"` the pipeline streams the agent's reply (`stream_mode="messages"`), cuts it into sentences, and sends the first one to TTS while the agent is still writing the rest. First the effect with simulated timings, so it is visible without a key:

In [ ]:
# 🧪 Offline: SIMULATED agent (0.6 s to think, 60 tokens/s) and speaker (0.3 s + 4 ms/char).
from mastery.voice import ScriptedAgent

REPLY = ("I can see the export errors on your account. Our status page shows a degraded station sync since 9 a.m., "
         "and engineers are on it. I will email you as soon as it works again, usually within the hour.")
rows = []
for mode in ("sequential", "sentences"):
    sim = VoicePipeline(fake_stt, ScriptedAgent(REPLY), [FakeSpeaker(seconds_per_request=0.3, seconds_per_char=0.004, max_chars=2000)])
    t = sim.run(None, key="q-status", mode=mode)
    rows.append({"mode": mode, **t.timings, "chunks": t.speech.chunks})
table(rows, ["mode", "stt", "agent", "tts", "first_audio", "total", "chunks"])

**You should now see** first audio after about **2.4 s** in sequential mode and about **1.2 s** in sentence mode, while the total gets slightly *longer* (three TTS requests instead of one).

### What just happened?

The caller hears the first sentence while the rest is still being generated: first audio halves, the total doesn't matter as long as the next sentence is ready before the first finishes playing (a 9-word sentence plays for about 3 seconds). Short per-sentence requests are also exactly what Orpheus's 200-character limit wants, so the live version uses Orpheus:

In [ ]:
# 💸 (LIVE) One call, sequential vs sentences, Orpheus as the speaker (about 4 Orpheus requests).
if LIVE:
    orpheus_only = [OrpheusSpeaker(budget=ORPHEUS_BUDGET)]
    rows = []
    for mode in ("sequential", "sentences"):
        live = VoicePipeline(stt, GraphAgent(live_desk), orpheus_only, stt_options=HINTS, clean_transcript=fix_ids)
        t = live.run(clips["q-status"], mode=mode)
        rows.append({"mode": mode, **t.timings, "chunks": t.speech.chunks})
        if any("model_terms_required" in e for e in t.speech.errors):
            print("⚠️ Orpheus fell back to text: model_terms_required. Accept the terms once at", ORPHEUS_TERMS_URL)
            break
    table(rows, ["mode", "stt", "agent", "tts", "first_audio", "total", "chunks"])
    print("Orpheus budget left:", ORPHEUS_BUDGET.remaining())
else:
    print("⏭️ Skipped: needs GROQ_API_KEY and the Part 2 recordings (see the setup cell).")

> 💸 Two Whisper requests, about 4–8 `gpt-oss-20b` calls, about 4 Orpheus requests.

**You should now see** `first_audio` lower in `sentences` mode, usually by around a second. If it isn't, look at `agent`: when the model writes the whole reply in one burst after a long tool step, there is nothing to overlap. Record both rows in the research log.

---
## Part 6 · Who is calling? Identity from the phone system, not from speech

"This is customer C-1002" is just audio. Anyone can say it, and Whisper may mishear it. The phone system knows the **calling number**: map it to a customer and pass that as the desk's `SupportContext` (Project 09). Then customer-scoped tools act only on that account, whatever the caller says.

In [ ]:
# 🧪 Offline: caller-ID lookup, then the same request with and without a signed-in context.
PHONE_BOOK = {"+57 300 555 0142": "C-1002", "+57 315 555 0101": "C-1001"}   # from your CRM, not from the call

question = CALLS["q-unlock"]["text"]
for label, context in (("no caller ID (demo)", None),
                       ("caller ID → C-1001", SupportContext(PHONE_BOOK[CALLS["q-unlock"]["caller_number"]]))):
    reply = GraphAgent(offline_desk, context=context).invoke(question)
    print(f"{label:<20} → {reply[:110]}")

**You should now see** without a caller ID `Account C-1002 was not locked.` (the tool trusted the spoken ID), and with the caller identified as C-1001 `Not allowed: you are signed in as C-1001, so I can only act on C-1001's account, not C-1002.`

### What just happened?

The call came from C-1001's number while the caller *said* C-1002. With the context set, the desk refused to act on the other account. For sensitive actions (refunds, email changes) add a second factor the caller can't just say: a one-time code sent by SMS to the number on file.

---
## Part 7 · This version vs the official streaming voice agent

| | This lesson (batch) | [LangChain voice agent tutorial](https://docs.langchain.com/oss/python/langchain/voice-agent) |
|---|---|---|
| Input | Whole recording, then STT | Microphone audio streamed over a WebSocket to streaming STT |
| End of question | The file ends | Voice activity detection / endpointing (Lesson 13.05) |
| Agent | `create_agent` desk, streamed by sentence | `create_agent`, streamed |
| TTS | One request per sentence | Streaming TTS, audio chunks played as they arrive |
| Interruptions (barge-in) | None | Caller can talk over the agent; playback stops |
| Providers | Groq Whisper, Gemini / Orpheus | AssemblyAI, Cartesia (others possible) |
| Typical first audio | 1.5–4 s | Under 1 s (their claim: under 700 ms) |

The missing pieces are the subject of Lesson 13.05: endpointing, streaming STT/TTS, barge-in, and telephony.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Caller hears file paths or "hash hash" | Agent reply sent to TTS as is | `for_voice` (citations, headings, 3 sentences) |
| Long silence before the answer | Sequential mode waits for the whole reply and clip | `mode="sentences"`; keep the voice prompt short |
| Answer about the wrong account | Spoken ID trusted | Caller ID → `SupportContext`; second factor for sensitive actions |
| First sentence is "Sure!" and nothing else for seconds | Sentence split on a tiny fragment | `sentences(min_chars=20)` merges short starts |
| Orpheus 429 halfway through a call | Daily quota | `TtsBudget`; Gemini first in the chain; text fallback |
| Orpheus answers `400 model_terms_required` | Terms not accepted for your Groq org | Accept once in the Groq playground (Lesson 13.03, Step 0); the chain falls through meanwhile |
| Stream yields nothing | Tool-call chunks only, final answer not streamed | Check `stream_mode="messages"` and filter on AI message chunks (as `GraphAgent.stream` does) |

## ⚠️ Production notes

> ⚠️ **Production note: trace a voice turn as one run.** (Parts 4–5)
> Log STT, agent and TTS as children of one trace with their timings, the transcript and the spoken text (not the audio, unless you have consent and a retention policy). Without it you can't tell whether a slow call was the model or the speech provider. See *Voice tracing fundamentals* in the reading list.

> ⚠️ **Production note: identity and consent on a phone line.** (Part 6)
> Identify callers by the number and a second factor, never by what they say. Announce that the call is handled by an automated assistant and may be recorded, and give a way to reach a person.

## 🏋️ Exercises

1. **(Easy)** Change `max_sentences` to 1 and to 5 for the `q-status` call. How do spoken length and `tts` time change?
2. **(Medium)** Add a `confirm` step: when the transcript contains a customer ID that differs from the caller-ID customer, answer "I can only help with the account linked to this phone number" without calling the agent.
3. **(Medium)** Wrap the three stages with LangSmith's `@traceable` (names `stt`, `agent`, `tts`) so one call shows as one trace with three children. Run it with `LANGSMITH_TRACING=true`.
4. **(Stretch)** In sentence mode TTS runs between sentences and blocks the agent stream. Run TTS in a background thread (`concurrent.futures.ThreadPoolExecutor`) so the agent keeps streaming. Measure the total.

<details>
<summary><b>Show solutions</b></summary>

**1.** `VoicePipeline(fake_stt, ScriptedAgent(REPLY), [FakeSpeaker(seconds_per_request=0.3, seconds_per_char=0.004, max_chars=2000)], max_sentences=1)`. Fewer sentences means less to listen to and a shorter TTS stage; for voice, one or two sentences plus "do you want more detail?" usually beats a paragraph.

**2.**
```python
import re

def guarded_turn(audio, key, caller_customer):
    heard = fake_stt.transcribe(audio, key=key)
    said = {f"C-{n}" for n in re.findall(r"c[\s-]?(\d{4})", heard.text.casefold())}
    if said and said != {caller_customer}:
        return "I can only help with the account linked to this phone number."
    return GraphAgent(offline_desk, context=SupportContext(caller_customer)).invoke(heard.text)

print(guarded_turn(None, "q-unlock", "C-1001"))
```

**3.**
```python
from langsmith import traceable

class TracedPipeline(VoicePipeline):
    @traceable(name="voice_turn")
    def run(self, audio, **kw):
        return super().run(audio, **kw)

stt_t = traceable(name="stt")(fake_stt.transcribe)
```
Wrap each stage the same way (`traceable(name=...)` on `transcribe`, `invoke`/`stream`, `speak`), then open the project `m13-voice-agent-sandwich` in LangSmith.

**4.** Submit each sentence's `speak_with_fallback` to an executor, keep the futures in order, and stitch `[f.result().wav for f in futures]` at the end. The first future's completion time is your first audio; the total drops to roughly agent time + one TTS request.
</details>

## 🔎 Docs drill (5 min)

In LangChain's voice agent tutorial, which three stages make up the sandwich, and which transport connects the browser to the backend?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Build a voice agent with LangChain](https://docs.langchain.com/oss/python/langchain/voice-agent) and read the architecture section.
2. The stages are **speech-to-text → agent → text-to-speech**, each streamed (the tutorial uses AssemblyAI and Cartesia).
3. The browser streams audio to the backend over a **WebSocket**, and receives audio back the same way.

</details>

## ✅ Checkpoint

- [ ] I ran a voice turn offline and live, and know which stage is slowest.
- [ ] I measured first audio in sequential and sentence mode.
- [ ] I shortened agent replies for listening with `for_voice`.
- [ ] I identified the caller from the phone system and saw the desk refuse a spoken ID.
- [ ] I can list what the batch sandwich leaves out compared with a streaming voice agent.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read the [voice agent tutorial](https://docs.langchain.com/oss/python/langchain/voice-agent) again and log in [`docs/research-log.md`](../../docs/research-log.md): its providers today, its latency claim, and your measured first-audio times from Parts 4–5 (sequential and sentences). Note which stage dominated.

## Summary

- A voice turn is STT → agent → TTS; what the caller feels is end of question → first audio.
- Sequential mode waits for everything; sentence streaming speaks while the agent writes and roughly halves first audio.
- Shorten replies for listening; three sentences is about 15 seconds.
- Identify callers by phone number (and a second factor), never by a spoken ID.
- The batch sandwich is a learning tool; real-time agents stream every stage (Lesson 13.05).

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 13-05-voice-architecture.ipynb** → [`13-05-voice-architecture.ipynb`](13-05-voice-architecture.ipynb).

[Module 13 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)